# Multimodal Model Compression Framework - Demo

End-to-end walkthrough of the core compression pipeline via Python API.

1. Load CLAP model
2. Evaluate baseline
3. Compress (quantize + prune)
4. Benchmark PyTorch + ONNX Runtime
5. Review generated reports

> **Prerequisite**: run `pip install -e ..` from the project root, or ensure `src/` is importable.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path when running from notebooks/
sys.path.insert(0, str(Path("..").resolve()))

import torch
from omegaconf import DictConfig

from src.models import load_model
from src.compression import compress_model
from src.evaluation import evaluate_model, benchmark_pytorch_inference, benchmark_onnx_runtime

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")

## 2. Load Model

In [ ]:
model_cfg = DictConfig({"name": "laion/clap-htsat-fused"})
model = load_model(model_cfg)

n_params = sum(p.numel() for p in model.model.parameters())
print(f"Model: {model_cfg.name}")
print(f"Parameters: {n_params:,}")
print(f"Device: {model.device}")

## 3. Evaluate Baseline (Synthetic)

In [ ]:
dataset_cfg = DictConfig({"name": "hf-internal-testing/librispeech_asr_dummy", "split": "validation"})
eval_cfg = DictConfig({"batch_size": 1, "synthetic_runs": 10})

baseline_metrics = evaluate_model(model, dataset_cfg, eval_cfg)
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v}")

## 4. Compress Model (Quantization + Pruning)

In [ ]:
compression_cfg = DictConfig({
    "quantization": {"enabled": True, "dtype": "qint8"},
    "pruning":      {"enabled": True, "amount": 0.3},
    "distillation": {"enabled": False},
    "triton":       {"enabled": False},
})

compressed_model = compress_model(model, compression_cfg)

n_zeros = sum(
    (p == 0).sum().item()
    for p in compressed_model.model.parameters()
    if p.is_floating_point()
)
n_total = sum(
    p.numel()
    for p in compressed_model.model.parameters()
    if p.is_floating_point()
)
print(f"Sparsity after pruning: {100 * n_zeros / max(n_total, 1):.1f}%")
print("Compression complete.")

## 5. Benchmark Compressed Model

In [ ]:
benchmark_cfg = DictConfig({
    "batch_size": 1,
    "warmup_runs": 3,
    "benchmark_runs": 10,
    "onnx": {
        "enabled": True,
        "export_path": "../experiments/onnx/demo_model.onnx",
        "opset": 17,
        "allow_float_export_fallback": True,
        "warmup_runs": 3,
        "benchmark_runs": 10,
        "prefer_cuda": False,
    },
})

pt_metrics  = benchmark_pytorch_inference(compressed_model, benchmark_cfg)
ort_metrics = benchmark_onnx_runtime(compressed_model, benchmark_cfg)

all_metrics = {**pt_metrics, **ort_metrics}
print("Benchmark results:")
for k, v in all_metrics.items():
    print(f"  {k}: {v}")

## 6. Visualize Experiment Reports

Load JSON reports from `experiments/reports/` and compare PyTorch vs ONNX latency across runs.

In [ ]:
import json
from pathlib import Path
import plotly.graph_objects as go

reports_dir = Path("../experiments/reports")
report_files = sorted(reports_dir.glob("*.json")) if reports_dir.exists() else []

if not report_files:
    print("No reports found. Run: multimodal-compress benchmark --config configs/benchmark_config.yaml")
else:
    rows = []
    for f in report_files:
        data = json.loads(f.read_text(encoding="utf-8"))
        data["run"] = f.stem
        rows.append(data)

    runs   = [r["run"] for r in rows]
    pt_lat = [r.get("pytorch_latency_ms", None) for r in rows]
    ort_lat= [r.get("onnx_latency_ms", None) for r in rows]

    fig = go.Figure()
    fig.add_trace(go.Bar(name="PyTorch (ms)",    x=runs, y=pt_lat))
    fig.add_trace(go.Bar(name="ONNX Runtime (ms)", x=runs, y=ort_lat))
    fig.update_layout(
        title="Inference Latency per Benchmark Run",
        xaxis_title="Run",
        yaxis_title="Latency (ms)",
        barmode="group",
    )
    fig.show()
    print(f"Loaded {len(rows)} report(s).")

## 7. CLI Quick Reference

Run the same pipeline from the terminal:

```bash
# Analyze baseline
multimodal-compress analyze --config configs/clap_config.yaml

# Compress (quantize + prune)
multimodal-compress compress --config configs/compression_config.yaml

# Finetune compressed model
multimodal-compress finetune --config configs/finetune_config.yaml

# Benchmark (PyTorch + ONNX Runtime) and write reports
multimodal-compress benchmark --config configs/benchmark_config.yaml
```

Or use FastAPI:

```bash
uvicorn src.ui.api:app --reload
# POST /compress   with a YAML config file
# POST /finetune   with a YAML config file
# POST /benchmark  with a YAML config file
# GET  /health
```